# Citi Bike pipeline: data-quality evidence

This notebook reads the current Iceberg tables and produces a board-ready summary of the Bronze-to-Silver WAP result.

In [3]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from jobs.ml_utils import create_spark_session, get_catalog_name, load_config

cfg = load_config()
catalog = get_catalog_name(cfg)
spark = create_spark_session(cfg, app_name="CitiBikeDataQualityEvidence")

bronze_table = f"{catalog}.bronze.citibike_raw"
silver_table = f"{catalog}.silver.citibike_trips"
quarantine_table = f"{catalog}.quarantine.citibike_trips"

print("Reading current pipeline tables:")
print(f"  {bronze_table}")
print(f"  {silver_table}")
print(f"  {quarantine_table}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/04 18:36:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Reading current pipeline tables:
  nyc.bronze.citibike_raw
  nyc.silver.citibike_trips
  nyc.quarantine.citibike_trips


In [4]:
from pyspark.sql import functions as F

bronze_result = (
    spark.table(bronze_table)
    .agg(
        F.count("*").alias("raw_rides"),
        F.countDistinct("source_filename").alias("monthly_files"),
    )
    .first()
)
raw_rides = int(bronze_result["raw_rides"])
monthly_files = int(bronze_result["monthly_files"])
trusted_rides = spark.table(silver_table).count()
quarantined_rides = spark.table(quarantine_table).count()
quarantine_rate = 100 * quarantined_rides / raw_rides if raw_rides else 0.0

assert raw_rides == trusted_rides + quarantined_rides, (
    "Reconciliation failed: Bronze must equal Silver plus quarantine."
)
print("Reconciliation passed: raw = trusted + quarantined")

Reconciliation passed: raw = trusted + quarantined


In [6]:
import pandas as pd
from IPython.display import Markdown, display

summary = pd.DataFrame(
    {
        "Evidence": [
            "Monthly source files",
            "Raw rides received (Bronze)",
            "Trusted rides published (Silver)",
            "Rows quarantined",
            "Quarantine rate",
        ],
        "Value": [
            f"{monthly_files:,}",
            f"{raw_rides:,}",
            f"{trusted_rides:,}",
            f"{quarantined_rides:,}",
            f"{quarantine_rate:.3f}%",
        ],
    }
)
display(summary.style.hide(axis="index"))

statement = (
    f"Across the **{monthly_files} monthly files**, the pipeline received "
    f"**{raw_rides:,} raw rides**. It published "
    f"**{trusted_rides:,} trusted rides** and quarantined "
    f"**{quarantined_rides:,} rows**—about **{quarantine_rate:.3f}%**."
)
display(Markdown(statement))

Evidence,Value
Monthly source files,17
Raw rides received (Bronze),"60,307,939"
Trusted rides published (Silver),"60,258,767"
Rows quarantined,"49,172"
Quarantine rate,0.082%


Across the **17 monthly files**, the pipeline received **60,307,939 raw rides**. It published **60,258,767 trusted rides** and quarantined **49,172 rows**—about **0.082%**.